# Part 3: UdaPlay Agent Demonstration

This notebook demonstrates the execution of the UdaPlay AI Research Agent, testing its RAG capabilities, web search fallback, and conversational memory.

In [1]:
import logging
import json
import os
import sys

# 1. Add the parent directory to the system path so Jupyter can find 'src'
sys.path.append(os.path.abspath('..'))

# Configure logging to see the agent's thought process in the notebook
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s - %(levelname)s - %(message)s')

from src.agent import UdaPlayAgent

agent = UdaPlayAgent()
print("UdaPlay Agent successfully initialized!")

2026-03-28 20:40:41,097 - INFO - Initialized ChromaDB collection: 'games' at 'vector_db'
UdaPlay Agent successfully initialized!


### Helper Function
To format and cleanly print the output responses from the agent.

In [2]:
def print_agent_response(result):
    print("\n" + "="*50)
    print("FINAL ANSWER (Natural Language):")
    print("-"*50)
    print(result['final_answer'])
    print("\n" + "="*50)
    print("STRUCTURED DATA (JSON):")
    print("-"*50)
    print(json.dumps(result['structured_response'], indent=2))
    print("="*50 + "\n")

### Test Query 1: Internal Database Retrieval
Smoke tests a question about game information that should be present in the local ChromaDB.

In [4]:
query1 = "What platforms is 'Cyberpunk 2077' available on, and what was its original release date?"
print(f"User: {query1}")
result1 = agent.invoke(query1)
print_agent_response(result1)

User: What platforms is 'Cyberpunk 2077' available on, and what was its original release date?
2026-03-28 20:43:21,818 - INFO - 
--- Processing Query: 'What platforms is 'Cyberpunk 2077' available on, and what was its original release date?' ---
2026-03-28 20:43:21,821 - INFO - Executing Node: retrieve_node
2026-03-28 20:43:21,822 - INFO - Retrieving games for query: 'What platforms is 'Cyberpunk 2077' available on, and what was its original release date?'
2026-03-28 20:43:24,835 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-28 20:43:24,844 - INFO - Executing Node: evaluate_node
2026-03-28 20:43:24,845 - INFO - Evaluating retrieval sufficiency...
2026-03-28 20:43:29,014 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 20:43:29,018 - INFO - Evaluation result: Sufficient=True, Reasoning: The retrieved context provides the original release date of 'Cyberpunk 2077' as December 10, 2020, and list

BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for response_format 'FinalAnswerResponse': In context=('properties', 'structured_data'), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}

### Test Query 2: Web Search Fallback (Latest News)
This asks a question about the latest news, which should trigger the evaluator to determine the internal database lacks sufficient context, forcing a web search.

In [5]:
query2 = "What is the latest news regarding the release date of the game Grand Theft Auto 6?"
print(f"User: {query2}")
result2 = agent.invoke(query2)
print("Response from agent:")
print_agent_response(result2)

User: What is the latest news regarding the release date of the game Grand Theft Auto 6?
2026-03-28 20:45:04,090 - INFO - 
--- Processing Query: 'What is the latest news regarding the release date of the game Grand Theft Auto 6?' ---
2026-03-28 20:45:04,093 - INFO - Executing Node: retrieve_node
2026-03-28 20:45:04,094 - INFO - Retrieving games for query: 'What is the latest news regarding the release date of the game Grand Theft Auto 6?'
2026-03-28 20:45:05,034 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-28 20:45:05,047 - INFO - Executing Node: evaluate_node
2026-03-28 20:45:05,048 - INFO - Evaluating retrieval sufficiency...
2026-03-28 20:45:09,187 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 20:45:09,192 - INFO - Evaluation result: Sufficient=False, Reasoning: The retrieved context is entirely focused on the game Cyberpunk 2077 and does not contain any information regarding Grand Th

c:\Users\andres.leonrangel\ws\gh-aleon1220\udaplay-ai-research-agent\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:522: UserWarning: Invalid schema for OpenAI's structured output feature, which is the default method for `with_structured_output` as of langchain-openai==0.3. Specify `method="function_calling"` instead or update your schema. See supported schemas: https://platform.openai.com/docs/guides/structured-outputs#supported-schemas
  warnings.warn(message)


BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for response_format 'FinalAnswerResponse': In context=('properties', 'structured_data'), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}

### Test Query 3: Conversational Memory
We ask a follow-up question referencing the previous query to test the agent's stateful memory (LangGraph state/history).

In [6]:
query3 = "Who is the developer of that game?"
print(f"User: {query3}")
result3 = agent.invoke(query3)
print_agent_response(result3)

User: Who is the developer of that game?
2026-03-28 20:49:12,601 - INFO - 
--- Processing Query: 'Who is the developer of that game?' ---
2026-03-28 20:49:12,603 - INFO - Executing Node: retrieve_node
2026-03-28 20:49:12,604 - INFO - Retrieving games for query: 'Who is the developer of that game?'
2026-03-28 20:49:13,515 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-28 20:49:13,529 - INFO - Executing Node: evaluate_node
2026-03-28 20:49:13,530 - INFO - Evaluating retrieval sufficiency...
2026-03-28 20:49:19,034 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 20:49:19,037 - INFO - Evaluation result: Sufficient=False, Reasoning: The retrieved context does not provide the name of the developer for the game in question. While it mentions the game 'Cyberpunk 2077' and 'Grand Theft Auto VI', it does not specify the developers for either game. Therefore, it lacks sufficient information to answer t

BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for response_format 'FinalAnswerResponse': In context=('properties', 'structured_data'), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}